## Module score

In [1]:
# Path and system utilities
import os                    # Operating system interface
import sys                   # System-specific parameters and functions
import glob                  # File pattern matching
from pathlib import Path     # Object-oriented filesystem paths
from pyhere import here      # Reproducible project paths

# Single-cell data handling
import anndata as ad            # Core data structure for single-cell data
import scanpy as sc          # Analysis and visualization of single-cell data
import pyucell as uc

# parallel processing
from joblib import parallel_backend  # Parallel computing support

# dataframes
import pandas as pd

# Miscellaneous utilities
import warnings              # Suppress or manage warnings

# Custom modules and functions
sys.path.append(str(here('scripts/misc')))  # Add custom script path to system
import my_anndata as ma                    # Custom AnnData utilities

Parameters

In [2]:
# Paths
base_dir = str(here('data/annotate/'))
plot_dir = os.path.join(base_dir, 'plot') 
files_dir = os.path.join(base_dir, 'files') 

anndata_dir = str(here('data/anndata/'))
harmo_dir = Path(here('data/marker_database/harmonized'))

Load data

In [3]:
adata = ad.read_h5ad(os.path.join(anndata_dir, "AG_combined.h5ad"))

In [4]:
# path to genelists
genelists_path = glob.glob(os.path.join(harmo_dir, '*.csv'))

genelists = {}

for file in genelists_path:
    name = Path(file).stem.replace('_genes', '')
    df = pd.read_csv(file, sep=",", dtype=str)
    df_grouped = df.groupby('cell_type')['gene'].apply(list)
    for celltype, genes in df_grouped.items():
        key = f"{name}_{celltype}"
        genelists[key] = genes
            

In [5]:
uc.compute_ucell_scores(adata, signatures=genelists, n_jobs= 60)

Save data

In [6]:
adata.obs[adata.obs.columns[adata.obs.columns.str.endswith('UCell')]].to_csv(os.path.join(files_dir, "marker_gene_scores.csv"), index_label='barcode')